# 16 · Workflows – agents that follow a plan

One agent with many tools gets confused. A **workflow** splits the job into steps, each with a focused agent, and defines **who goes next**. This chapter uses **Microsoft Agent Framework**, Foundry's recommended way to build workflows (the portal's visual designer is retired after 2026-12-01).

The key design question is **who picks the next step**:

| Pattern | Who picks the next step | Use for | Section |
|---|---|---|---|
| **Sequential** (A → B → C) | **your code**: fixed edges | pipelines, document processing | 1 · invoice pipeline, with a threshold branch |
| **Concurrent** (fan out, merge) | **your code**: everyone at once, then an aggregator | briefings, gather and summarise | 2 · morning briefing |
| **Group chat** | a **selector** function (or an orchestrator LLM) | review boards, debate | 3 · review board |
| **Handoff** | the **agents**, only along the hand-offs you allow | triage, specialists | 4 · workshop support |
| **Declarative YAML** | your code: the same fixed flow, as a file | reviewable, versioned flows | 5 · invoice pipeline as YAML |
| Magentic | a manager agent | open-ended tasks | not covered |

In every pattern a model decides what to *say* and which *tools* to call **inside** a step. With fixed edges it never decides which step comes next.

Sections 1–5 run **in your Python process**, calling Foundry models. Section 6 hosts a workflow **in Foundry** as a hosted agent. Deep dive: [docs/workflows.md](../docs/workflows.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *
from stark_docs import *
import stark_docs
from stark_workflows import *   # runs a workflow and shows it as Markdown: plumbing only

fix_dotnet_for_powerfx()   # only changes anything for an x64 Python on Windows-on-ARM (needed in section 5)

from typing import Annotated
from pydantic import Field
from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from agent_framework.orchestrations import SequentialBuilder, ConcurrentBuilder, GroupChatBuilder, GroupChatState, HandoffBuilder

client = FoundryChatClient(project_endpoint=cfg("STARK_PROJECT_ENDPOINT"), model=model("chat"), credential=credential())

## Tools and tasks
The pipeline's tools are plain Python functions. `approval_mode="never_require"` lets the agent call them without a human (fine for read-only tools). The logic itself comes from [stark_docs.py](stark_docs.py), the same functions as in chapter 13.

The reviewer follows the **invoice-review skill** from chapter 15, here as plain instructions (direct injection, chapter 07): `skill_body()` returns its SKILL.md without the front matter. This chapter gives the agents no shell, so the skill's scripts do not run. The skill says what to do then: run checks 1 to 3 and 7 itself, and say that checks 4 to 6 (vendor master, IBAN, duplicates) were not checked.

The tasks sent to the workflows are named variables, and every run below shows the task it got:

| Variable | Task | Used in |
|---|---|---|
| `pym_task` | process `invoice-pym-4471.pdf`, about 3,332.34 USD | sections 1, 5 and 6 |
| `oscorp_task` | process `invoice-oscorp-9930.pdf`, about 99,792.00 USD | section 1 |
| `briefing_task` | ask J.A.R.V.I.S. for the morning briefing | sections 2 and 6 |

In [ ]:
@tool(approval_mode="never_require")
def extract_invoice_fields(path: Annotated[str, Field(description="Invoice file, e.g. data/docs/invoice-pym-4471.pdf")]) -> str:
    """Read an invoice with Document Intelligence and return fields, line items and confidences as JSON."""
    import json
    return json.dumps(extract_invoice(path), default=str)

@tool(approval_mode="never_require")
def check_approval_limit(amount: Annotated[float, Field(description="Invoice total in USD")],
                         approver: Annotated[str, Field(description="e.g. 'pepper potts'")]) -> str:
    """Check whether an amount is within the approver's limit."""
    import json
    return json.dumps(stark_docs.check_approval_limit(amount, approver))

invoice_review_skill = skill_body("invoice-review")   # SKILL.md without its front matter (direct injection, chapter 07)

pym_task = "Process the invoice at data/docs/invoice-pym-4471.pdf"        # about 3,332.34 USD
oscorp_task = "Process the invoice at data/docs/invoice-oscorp-9930.pdf"  # about 99,792.00 USD
briefing_task = "Good morning, J.A.R.V.I.S. Brief me."

## 1 · Sequential: the invoice pipeline
Three specialists. The reviewer gets the **invoice-review skill** as instructions and `check_approval_limit` as its tool. `intermediate_output_from` makes the first two steps visible, not only the last.

> ℹ️ `SequentialBuilder` is the shortcut: the list order is the run order. Below, the **same chain is wired by hand** with `WorkflowBuilder`; the order is fixed either way.

In [ ]:
extractor = client.as_agent(name="extractor", tools=[extract_invoice_fields],
    instructions="Extract the invoice. Call extract_invoice_fields with the path from the user, then list vendor, number, date, total and any low-confidence (<0.8) field.")
reviewer = client.as_agent(name="reviewer", tools=[check_approval_limit],
    instructions="You review the extracted invoice. Follow this checklist exactly:\n" + invoice_review_skill +
                 "\nUse check_approval_limit with approver 'pepper potts'.")
reporter = client.as_agent(name="reporter",
    instructions="Write a three-bullet message for Pepper Potts: what was reviewed, the decision, what she must do next.")

def build_pipeline():
    # a workflow keeps its conversation between runs, so build a fresh one for every run
    return SequentialBuilder(participants=[extractor, reviewer, reporter], intermediate_output_from=[extractor, reviewer]).build()

### Step 1 · The Pym invoice

`show_run()` ([stark_workflows.py](stark_workflows.py)) runs a workflow with streaming and shows the run as Markdown: the task, what each step said, in order, and a check table. The output looks like this (shortened):

```text
Task
> Process the invoice at data/docs/invoice-pym-4471.pdf
step · extractor     vendor, number, date, total, low-confidence fields
step · reviewer      the checklist, the limit check, a verdict
final · reporter     three bullets for Pepper Potts
Check                the steps, the reviewer's last verdict keyword, the time
```

**Goal:** the baseline: a fixed chain of three agents on a small invoice.

| Input | Value |
|---|---|
| task | `pym_task` |
| approver | Pepper Potts (limit 250,000 in `check_approval_limit`) |

**Expected:** three steps in this order: `extractor`, `reviewer`, `reporter`. The extractor names Pym Particle Supply, invoice 4471 and a total of about 3,332.34 USD. The reviewer calls `check_approval_limit`, which finds the total within Pepper's limit, and says that checks 4 to 6 were not run. ❓ The verdict is `APPROVE` or `NEEDS HUMAN REVIEW`: nothing on the invoice is wrong, but without the scripts the model judges low-confidence fields on its own (chapter 15 saw the total read at confidence 0.44 in one test run).

In [ ]:
await show_run(build_pipeline(), pym_task, expected=("APPROVE", "NEEDS HUMAN REVIEW"))

### Step 2 · Another invoice, the same path

**Goal:** fixed edges: a different invoice takes exactly the same three steps. Only the invoice changes.

| Input | Value |
|---|---|
| task | `oscorp_task` |
| approver | Pepper Potts (limit 250,000) |

**Expected:** the same three steps as in step 1. 99,792 USD is within Pepper's limit, so `check_approval_limit` passes again. In chapter 15 the agent rejected this invoice, because `vendor_check.py` found Oscorp blocked in the vendor master. Here the reviewer has neither the vendor master nor a shell, so it can only say that check 4 was not run. ❓ The verdict is `APPROVE` or `NEEDS HUMAN REVIEW`. The router below changes the *path* instead.

> ⚠️ Build a **new workflow for every run** (`build_pipeline()`). A workflow object keeps its conversation, and running the same object a second time fails with `No tool call found for function call output`. The agents can be reused.

In [ ]:
await show_run(build_pipeline(), oscorp_task, expected=("APPROVE", "NEEDS HUMAN REVIEW"))

### Step 3 · The same chain, wired by hand
`SequentialBuilder(participants=[a, b, c])` is a shortcut for a **linear graph**. With `WorkflowBuilder` you write the graph yourself: wrap each agent in an `AgentExecutor` (a box) and join the boxes with `add_edge(from, to)` (an arrow). The order is hard-wired in code, and the model never changes it.

| | `SequentialBuilder` | `WorkflowBuilder` + `add_edge` |
|---|---|---|
| order | list order | every edge written out |
| what each agent sees | the conversation so far | your choice per step: `AgentExecutor(agent, context_mode=...)` with `full`, `last_agent` or `custom` |
| beyond a straight line | no | conditions on edges, fan-out, custom code steps |
| used in | section 1, the first runs | section 1 (this cell), and the **hosted** version in section 6 |

`output_from` marks the box whose result is the workflow's answer, and `intermediate_output_from` shows progress from the earlier boxes.

**Goal:** the same chain as a graph you write yourself.

**Expected:** the same three steps in the same order as in step 1, on the same task (`pym_task`). ❓ The verdict is again `APPROVE` or `NEEDS HUMAN REVIEW`.

In [ ]:
from agent_framework import AgentExecutor, WorkflowBuilder

def build_pipeline_with_edges():
    # the same three agents as above, now as boxes joined by arrows
    step1, step2, step3 = (AgentExecutor(a) for a in (extractor, reviewer, reporter))
    return (WorkflowBuilder(name="invoice_pipeline", start_executor=step1, output_from=[step3],
                            intermediate_output_from=[step1, step2])
            .add_edge(step1, step2)   # extractor -> reviewer
            .add_edge(step2, step3)   # reviewer  -> reporter
            .build())

await show_run(build_pipeline_with_edges(), pym_task, expected=("APPROVE", "NEEDS HUMAN REVIEW"))

Same agents, same fixed order. Use the shortcut for a plain chain. Reach for the explicit version when you need to control what each step sees, add a condition or a code step, or add a final step that returns plain text for hosting (section 6).

### Branch on a threshold: your code decides
The old Foundry workflow designer had condition nodes ("if the output says X, go to Y"). In Agent Framework the same thing is an **edge with a condition**: a plain Python function reads the previous step's output and picks the next step. No model is asked which branch to take.

The extractor answers in a fixed JSON shape (`response_format`), so the code can read the total as a number. Then the invoice goes one of three ways:

| Total (USD) | Next step | Who acts |
|---|---|---|
| up to 5,000, and no low-confidence field | `auto_approve` | code only, no model |
| above 50,000 | `escalate` | code only, hands it to Tony |
| anything else | `reviewer` → `reporter` | the agents from above |

| Piece | What it does here |
|---|---|
| `default_options={"response_format": InvoiceFacts}` | the extractor must answer as JSON that matches the Pydantic model `InvoiceFacts` |
| `add_switch_case_edge_group(extract, [Case(...), Case(...), Default(...)])` | checks the cases in order and sends the message to the **first** one whose condition is true; `Default` takes the rest |
| `@executor` | a step that is plain Python: it reads the facts and returns a decision without calling a model |
| `output_from=[auto_approve, escalate, report]` | every branch ends in a step whose result is the workflow's answer |
| `intermediate_output_from=[extract, review]` | shows the facts the conditions read and the reviewer's work, not only the answer |

The thresholds are arguments of `build_router()`, so you can change them per run and watch the path change. `show_route()` ([stark_workflows.py](stark_workflows.py)) runs the router and shows the facts (the JSON the conditions read), the answer and the `path`: the executors in the order they ran. Three cases follow. Cases B and C each change one thing compared with case A.

In [ ]:
from typing import Never
from pydantic import BaseModel
from agent_framework import AgentExecutorResponse, Case, Default, WorkflowContext, executor

class InvoiceFacts(BaseModel):
    vendor: str
    invoice_number: str
    total_usd: float
    low_confidence_fields: list[str]

facts_extractor = client.as_agent(name="facts_extractor", tools=[extract_invoice_fields],
    default_options={"response_format": InvoiceFacts},
    instructions="Call extract_invoice_fields with the path from the user. Return vendor, invoice number, total in USD "
                 "and the names of the fields with a confidence below 0.8.")

def facts(response: AgentExecutorResponse) -> InvoiceFacts:
    """The extractor's answer as typed data. The routing reads these fields, never free text."""
    return InvoiceFacts.model_validate_json(response.agent_response.text)

@executor(id="auto_approve")
async def auto_approve(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """A code step: no model is called."""
    f = facts(response)
    await ctx.yield_output(f"APPROVE (automatic): {f.vendor} {f.invoice_number}, {f.total_usd:,.2f} USD. No model reviewed it.")

@executor(id="escalate")
async def escalate(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """A code step: no model is called."""
    f = facts(response)
    await ctx.yield_output(f"ESCALATE to Tony Stark: {f.vendor} {f.invoice_number}, {f.total_usd:,.2f} USD is above the escalation threshold.")

def build_router(auto_up_to=5_000, escalate_above=50_000):
    # a workflow keeps its conversation between runs, so build a fresh one for every run
    extract, review, report = AgentExecutor(facts_extractor), AgentExecutor(reviewer), AgentExecutor(reporter)
    return (WorkflowBuilder(name="invoice_router", start_executor=extract, output_from=[auto_approve, escalate, report],
                            intermediate_output_from=[extract, review])
            .add_switch_case_edge_group(extract, [
                # checked in order, the first true condition wins
                Case(condition=lambda r: facts(r).total_usd <= auto_up_to and not facts(r).low_confidence_fields, target=auto_approve),
                Case(condition=lambda r: facts(r).total_usd > escalate_above, target=escalate),
                Default(target=review),   # everything in between gets the agent review
            ])
            .add_edge(review, report)
            .build())

### Case A · A small invoice, default thresholds

**Goal:** the cheap path: code approves a small invoice without a reviewer.

| Input | Value |
|---|---|
| task | `pym_task` (about 3,332.34 USD) |
| thresholds | `auto_up_to=5_000`, `escalate_above=50_000` (the defaults) |

**Expected:** ❓ it depends on what Document Intelligence returns. The facts in the output show which row applies:

| `low_confidence_fields` | Path | Answer |
|---|---|---|
| empty | `facts_extractor -> auto_approve` | `APPROVE (automatic): ...`, no model reviewed it |
| not empty (chapter 15 saw the total at confidence 0.44 in one test run) | `facts_extractor -> reviewer -> reporter` | the reviewer's checklist, then the reporter's three bullets |

Either way your code picked the path from the facts, not a model.

In [ ]:
await show_route(build_router(), pym_task)

### Case B · The same invoice, a lower threshold

**Goal:** change only the rule. The invoice is the one from case A.

| Input | Value |
|---|---|
| task | `pym_task` |
| thresholds | `auto_up_to=1_000`, `escalate_above=50_000` |

**Expected:** `facts_extractor -> reviewer -> reporter`, whatever the confidences: 3,332.34 is above 1,000 and not above 50,000, so the `Default` edge takes it.

In [ ]:
await show_route(build_router(auto_up_to=1_000), pym_task,
                 expected_path="facts_extractor -> reviewer -> reporter")

### Case C · A large invoice

**Goal:** the escalation branch. The default thresholds again, another invoice.

| Input | Value |
|---|---|
| task | `oscorp_task` (about 99,792 USD) |
| thresholds | the defaults |

**Expected:** `facts_extractor -> escalate`, and the answer `ESCALATE to Tony Stark: ...` with 99,792.00 USD. The reviewer and the reporter never run. Low-confidence fields make no difference: the first case fails on the total, and the escalation case is checked before `Default`.

In [ ]:
await show_route(build_router(), oscorp_task, expected_path="facts_extractor -> escalate")

The model still reads the invoice, but it does not choose the route. Your code does, so you can unit-test the rule without a model: the same facts always take the same path.

> ⚠️ If the extractor returns JSON that does not match `InvoiceFacts`, `facts()` raises and the run fails. That is the safe choice for money: an invoice the code cannot read is never auto-approved. To route it to a person instead, catch the error in a condition and send it to a review step.

## 2 · Concurrent: J.A.R.V.I.S.'s morning briefing
In a **concurrent** workflow every participant gets the **same input at the same time**, and an **aggregator** merges their answers. Three specialists check the workshop in parallel (suits, workshop systems, pending invoices); J.A.R.V.I.S. turns their reports into one briefing for Tony.

| Piece | What it does here |
|---|---|
| `suits`, `workshop`, `finance` | each calls the **stark-mcp** tools (chapter 06) through the project connection `stark-mcp`, with only the tools it needs |
| `ConcurrentBuilder(participants=[...])` | fans the request out to all three at once |
| `.with_aggregator(brief)` | runs when all three are done; the J.A.R.V.I.S. agent writes the merged briefing, and its text is the workflow's output |

`client.get_mcp_tool(..., project_connection_id="stark-mcp")` is a **hosted** MCP tool: Foundry calls the server and uses the connection's function key, so the key never reaches your code. The `store: False` option keeps the agents stateless, so the same code can be hosted in section 6.

In [ ]:
NO_STORE = {"store": False}

def stark(allowed):
    """The stark-mcp tools through the project connection (chapter 06). Each specialist only gets the tools it needs."""
    return client.get_mcp_tool(name="stark", url=cfg("STARK_MCP_URL"), project_connection_id="stark-mcp",
                               approval_mode="never_require", allowed_tools=allowed)

def build_briefing():
    # a workflow keeps its conversation between runs, so build a fresh one for every run
    suits = client.as_agent(name="suits", default_options=NO_STORE, tools=[stark(["list_suits", "get_suit"])], instructions=
        "You are the suit officer. Use your tools: list the suits, then check each one. Report which suits are ready for duty in two sentences.")
    workshop = client.as_agent(name="workshop", default_options=NO_STORE, tools=[stark(["get_workshop_status"])], instructions=
        "You are the workshop officer. Check the arc reactor, the fabrication bay and the hangar with your tool. Report in two sentences.")
    finance = client.as_agent(name="finance", default_options=NO_STORE, tools=[stark(["check_invoice_approval"])], instructions=
        "You are the finance officer. Two invoices wait for Pepper Potts: Pym Particle Supply 3,332.34 USD and Oscorp Materials 99,792.00 USD. "
        "Check both with your tool. Report in two sentences.")
    jarvis = client.as_agent(name="jarvis", default_options=NO_STORE, instructions=
        "You are J.A.R.V.I.S., dry and witty, addressing Tony Stark as sir. Merge the three reports into a morning briefing of at most five lines. "
        "Mention anything that needs his attention.")

    async def brief(results):
        reports = chr(10).join(f"[{r.executor_id}] {r.agent_response.text}" for r in results)
        return (await jarvis.run(reports)).text

    return (ConcurrentBuilder(name="morning_briefing", participants=[suits, workshop, finance],
                              intermediate_output_from=[suits, workshop, finance])
            .with_aggregator(brief).build())

### Run the briefing

`show_briefing()` ([stark_workflows.py](stark_workflows.py)) shows each specialist's report and J.A.R.V.I.S.'s merge. Its check table says how many seconds after the start each one sent its last words.

**Goal:** three specialists work at the same time, and one aggregator merges their reports.

| Input | Value |
|---|---|
| task | `briefing_task` |

**Expected:** three reports, from `suits`, `workshop` and `finance`, then J.A.R.V.I.S.'s briefing of at most five lines. The facts come from the static `stark-mcp` data: none of the four suits is ready for duty (museum, destroyed, destroyed, retired), the arc reactor is online, the fabrication bay is busy, the hangar is idle, and both invoices are within Pepper's limit of 250,000 USD. ❓ The three specialists finish close together, and the whole run takes about as long as the slowest one plus the merge (timings vary).

In [ ]:
await show_briefing(build_briefing(), briefing_task)

The three reports are collected **in whatever order they finish**: compare the finish times in the check table. The total time is about that of the slowest specialist plus the merge, not the sum of all four. The specialists never see each other's answers: only J.A.R.V.I.S. does.

> ℹ️ The data behind `stark-mcp` is static on purpose, so the briefing is the same every morning, apart from J.A.R.V.I.S.'s wording.

## 3 · Group chat: the review board
Agents share one conversation; a **selector** picks the next speaker (here round-robin; use `orchestrator_agent=` for an LLM that decides). A termination condition ends the chat.

`show_board()` ([stark_workflows.py](stark_workflows.py)) shows every turn in order and the last `DECISION:` of the security officer.

**Goal:** a selector function, not a model, picks the next speaker.

| Input | Value |
|---|---|
| question | `board_question`: 4 vials of Pym particles for 3,332 USD |
| selector | `round_robin`: the participant at `current_round % 3` |
| stop | six assistant messages |

**Expected:** the turns go `engineer`, `accountant`, `security`, then again in the same order, until six messages are written (❓ the exact count depends on where the round counter starts, so check the run's output). ❓ Whether `security` ends with `DECISION: approve` or `DECISION: reject` is the model's call.

In [ ]:
engineer = client.as_agent(name="engineer", instructions="You are a suit engineer. Judge technical need in two sentences.")
accountant = client.as_agent(name="accountant", instructions="You are the accountant. Judge cost and budget in two sentences.")
security = client.as_agent(name="security", instructions="You are the security officer. Judge supply-chain and safety risk in two sentences. If all three of you agree, end with 'DECISION: <approve|reject>'.")

def round_robin(state: GroupChatState) -> str:
    names = list(state.participants.keys())
    return names[state.current_round % len(names)]

board = GroupChatBuilder(
    participants=[engineer, accountant, security],
    termination_condition=lambda conversation: sum(1 for m in conversation if m.role == "assistant") >= 6,
    intermediate_output_from=[engineer, accountant, security],
    selection_func=round_robin,
).build()

board_question = "Should we buy 4 vials of Pym particles for 3,332 USD for the shrink-ray prototype?"
await show_board(board, board_question)

## 4 · Handoff: triage and specialists
In a **handoff**, agents decide to pass control based on the conversation; the user stays in the loop. `triage` routes each message to a specialist, the specialist answers, and control can go back to `triage`.

> ⚠️ Every participant must be created with `require_per_service_call_history_persistence=True`. Without it `build()` raises `Handoff workflows require all participant agents to have ...`. The flag keeps each agent's local history consistent with the service when control is handed over.

`enable_return_to_previous(False)` makes `triage` look at **every** user message. Without it, the next message goes straight to whoever spoke last.

In [ ]:
# handoff workflows need every participant to keep its history per service call
triage = client.as_agent(name="triage", require_per_service_call_history_persistence=True, instructions="Route the user: suits and hardware -> suit_engineer; access, badges, threats -> security_officer. Otherwise answer yourself.")
suit_engineer = client.as_agent(name="suit_engineer", require_per_service_call_history_persistence=True, instructions="You fix suits. Be practical. Hand back to triage when done.")
security_officer = client.as_agent(name="security_officer", require_per_service_call_history_persistence=True, instructions="You handle security incidents. Be strict. Hand back to triage when done.")

def build_handoff():
    # a workflow keeps its conversation between runs, so build a fresh one for every run
    return (
        HandoffBuilder(name="workshop_support", participants=[triage, suit_engineer, security_officer])
        .with_start_agent(triage)
        .enable_return_to_previous(False)
        .add_handoff(triage, [suit_engineer, security_officer])
        .add_handoff(suit_engineer, [triage])
        .add_handoff(security_officer, [triage])
        .build()
    )

print("handoff workflow built:", build_handoff().name)

### Run it with scripted user turns
A handoff workflow **pauses and asks the user** after each specialist answers: you get a `request_info` event with a `HandoffAgentUserRequest`. You answer it with `HandoffAgentUserRequest.create_response(text)`, or end the chat with `HandoffAgentUserRequest.terminate()`. `run_handoff()` ([stark_workflows.py](stark_workflows.py)) does that with a list of user messages instead of `input()`. It shows the transcript and a table of who answered each message.

**Goal:** the agents, not your code, pick the specialist, along the handoffs you allowed.

| User message in `support_turns` | Allowed route |
|---|---|
| the Mark 42 repulsor is jammed | `triage` → `suit_engineer` |
| a stolen badge opened the vault | `triage` → `security_officer` |
| that is all | `triage` |

**Expected:** ❓ the model decides. Usually `suit_engineer` answers the first message, `security_officer` the second and `triage` the third. `triage` may also appear before a specialist when it says something while handing over, and an agent may answer a question itself, so the route can vary between runs.

In [ ]:
support_turns = [
    "My Mark 42 repulsor is jammed and will not fire.",
    "Thanks. Now someone used a stolen badge to enter the vault yesterday.",
    "Thanks, that is all.",
]
await run_handoff(build_handoff(), support_turns)

For a real chat, read the next message with `input(...)` instead of taking it from a list.

## 5 · The same pipeline as YAML
Declarative workflows keep the *flow* in a reviewable file; agents are registered by name. It uses `extractor`, `reviewer` and `reporter` from section 1, so run that section's agent cell first. (Needs `agent-framework-declarative`, Python 3.10–3.13 and a **.NET runtime**: expressions like `=System.ConversationId` are evaluated by PowerFx. If you see `PowerFx is not available (dotnet runtime not installed)` although `dotnet` is installed, your Python and your .NET have different CPU architectures; `fix_dotnet_for_powerfx()` in the first cell handles the common x64-Python-on-ARM case. Restart the kernel after changing it.)

In [ ]:
print((ROOT / "workflows" / "invoice-pipeline.yaml").read_text())

**Goal:** the pipeline of step 1, with the flow read from the file instead of written in Python.

**Expected:** the three agents run in the order of the YAML on `pym_task`. `show_outputs()` ([stark_workflows.py](stark_workflows.py)) shows every output of the run as text. ❓ How many outputs the run returns, and which agent's text they carry, was not checked.

In [ ]:
from agent_framework.declarative import WorkflowFactory
factory = WorkflowFactory()
factory.register_agent("ExtractorAgent", extractor)
factory.register_agent("ReviewerAgent", reviewer)
factory.register_agent("ReporterAgent", reporter)
yaml_workflow = factory.create_workflow_from_yaml_path(ROOT / "workflows" / "invoice-pipeline.yaml")
yaml_result = await yaml_workflow.run(pym_task)
show_outputs(pym_task, yaml_result.get_outputs())

## 6 · Run it in Foundry
Foundry's own workflow designer is being retired (2026-12-01), and its YAML is a different dialect from the file in section 5. The supported route is to host your **Agent Framework** workflow as a **hosted agent**, like chapter 09: your code runs in a managed container, with an endpoint, an identity, versions and traces.

### Two finished projects

| Hosted agent | Pattern | What it needs |
|---|---|---|
| [invoice-workflow](../src/invoice-workflow/main.py) | the sequential chain of section 1, wired with `WorkflowBuilder` | Document Intelligence, so the agent's identity needs a role (see *Deploy*) |
| [morning-briefing](../src/morning-briefing/main.py) | the concurrent briefing of section 2 | the `stark-mcp` project connection from chapter 06, nothing else |

### What changes when you host a workflow

| In the hosted project | Why |
|---|---|
| `ResponsesHostServer(workflow=build_workflow, parse_response=...)` | hosts the workflow **natively**. `workflow.as_agent()` is deprecated, because a `WorkflowAgent` keeps state and must not serve different users |
| `build_workflow(request)` builds a **fresh** workflow per request | a workflow keeps its conversation (the same rule as `build_pipeline()` in section 1) |
| `default_options={"store": False}` on the agents | the host keeps the history; native workflows cannot checkpoint provider-side storage |
| the output is **plain text**: a final executor, or the aggregator's return value | raw streamed model updates cannot be a native workflow output (the host rejects them with `cannot checkpoint unfinished provider background output`) |

### Deploy

From a terminal in the repo root:

```bash
azd deploy invoice-workflow          # zips src/invoice-workflow, Foundry builds it, creates version 1
azd deploy morning-briefing          # same steps for the concurrent briefing
python scripts/grant_agent_role.py   # once, for invoice-workflow only: let its identity call Document Intelligence
```

`azd` first checks the installed extensions against `requiredVersions` in the root [azure.yaml](../azure.yaml). If you changed the `azure.ai.agents` extension for chapter 15, install the version that file asks for before you deploy.

> ⚠️ A deployed agent runs as **its own identity**, not yours. `invoice-workflow` fails with `PermissionDenied` until [grant_agent_role.py](../scripts/grant_agent_role.py) gives that identity the *Cognitive Services User* role (allow a minute or two to take effect). `morning-briefing` needs nothing: its MCP calls go through the project connection.

### Call them

A hosted agent is called like any agent, with a client **bound** to it (chapter 04). The bound client has its own conversations, so `ask_hosted()` ([stark_workflows.py](stark_workflows.py)) creates the conversation with the same client and deletes it afterwards:

```python
bound = openai_client(agent_name="invoice-workflow")
conversation = bound.conversations.create()
response = bound.responses.create(conversation=conversation.id, input=pym_task)
```

It shows the prompt, the whole answer and the time. Each call takes **30–40 seconds**: several model and tool calls per request. The two cells need the first cells of this notebook and the deploy above, not sections 1 to 5.

**Expected:** `invoice-workflow` answers with the reporter's three bullets for Pepper Potts about the Pym invoice. `morning-briefing` answers with J.A.R.V.I.S.'s briefing, from the same `stark-mcp` data as section 2.

In [ ]:
ask_hosted("invoice-workflow", pym_task)

In [ ]:
ask_hosted("morning-briefing", briefing_task)

> ⏱️ The first call after a container has been idle can be slower. If it answers `proxy_timeout` (HTTP 424, "agent container did not respond within the timeout window"), just call again.

## 👀 In the portal

**Build → Agents → invoice-workflow** and **morning-briefing**: versions, the playground, and **Traces** with one trace per call. `azd deploy` prints a direct link. They appear as hosted agents. ❓ Not checked: whether the portal draws the workflow graph (the local Agent Inspector in VS Code does).

## 🧹 Clean up
The hosted agents stay deployed. Uncomment the lines to remove them, or run `azd down --purge` for everything.

In [ ]:
# delete_agent("invoice-workflow")
# delete_agent("morning-briefing")

## 🎯 Recap

| Need | Pattern | Who picks the next step |
|---|---|---|
| Fixed steps | sequential | your code |
| Branch on the data | conditional edges (`Case` / `Default`) | your code |
| Fan out, then merge | concurrent + aggregator | your code |
| Debate / several viewpoints | group chat | a selector |
| Route to the right specialist, user in the loop | handoff | the agents |
| Reviewable flow in git | declarative YAML | your code |
| Run in Foundry | native hosting (`workflow=`) as a hosted agent | the same as the pattern you host |

> 💡 Next: [17 · Vision – the Avengers assemble](17_vision_a2a_capstone.ipynb) 🧪.